In [1]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

In [2]:
from src.pipeline.config import (
    TRAIN_DF_PATH,
	PROCESSED_DATA_DIR,
	RANDOM_SEED
)

In [3]:
df = pd.read_parquet(TRAIN_DF_PATH, engine="pyarrow")

In [4]:
df["TransactionDT"].is_monotonic_increasing

True

In [5]:
df["card2"] = df["card2"].fillna(-1)
df["addr1"] = df["addr1"].fillna(-1)

In [6]:
df["uid"] = df["card1"].astype(str) + "_" + df["card2"].astype("Int64").astype(str) + "_" + df["addr1"].astype("Int64").astype(str)

In [7]:
df["uid"].head(15)

0      13926_-1_315
1      2755_404_325
2      4663_490_330
3     18132_567_476
4      4497_514_420
5      5937_555_272
6     12308_360_126
7     12695_490_325
8      2803_100_337
9     17399_111_204
10     16496_352_-1
11      4461_375_-1
12     3786_418_204
13    12866_303_330
14    11839_490_226
Name: uid, dtype: str

In [ ]:
df["TransactionAmtMean"] = df.groupby("uid")["TransactionAmt"].transform("mean")
df["TransactionAmtMax"] = df.groupby("uid")["TransactionAmt"].transform("max")
df["TransactionAmtMin"] = df.groupby("uid")["TransactionAmt"].transform("min")
df["TransactionAmtStd"] = df.groupby("uid")["TransactionAmt"].transform("std").fillna(0)

df["TransactionAmt_Z"] = (df["TransactionAmt"] - df["TransactionAmtMean"]) / (df["TransactionAmtStd"] + 1e-5)
df["TransactionAmt_to_Mean"] = df["TransactionAmt"] / (df["TransactionAmtMean"] + 1e-5)

In [9]:
df["Hour"] = (df["TransactionDT"] // 3600) % 24
df["DayOfWeek"] = (df["TransactionDT"] // 86400) % 7
df["TransactionAmtLog"] = np.log1p(df["TransactionAmt"])